# Server Failure Prediction - TrustyAI Bias Test

This notebook tests the deployed server failure model with the TrustyAI bias metrics:

- **SPD** (statistical parity difference): difference in the rate of the favorable outcome between the unprivileged and the privileged group. Fair range: -0.1 to 0.1.
- **DIR** (disparate impact ratio): ratio of the same rates. Fair range: 0.8 to 1.25.

The protected attributes are the `workload_type_*` columns: `1` is the workload type of the column (unprivileged group), `0` is all other workload types (privileged group).
The favorable outcome is `0`, no failure predicted.

**Prerequisites:**

- You ran `train_model.ipynb`, which creates `test_data.csv` and uploads the serving model.
- The model is deployed with OpenVINO Model Server, in a project with the TrustyAI service.

## 1. Configuration

- `MODEL_NAME`: the name of the deployed model.
- `INFER_URL`: the inference endpoint of the model.
- `TRUSTY_ROUTE`: the TrustyAI service route, from `oc get route trustyai-service`.
- `TRUSTY_TOKEN`: your OpenShift token, from `oc whoami -t`.

In [ ]:
import time

import pandas as pd
import requests
import urllib3

import data_utils

urllib3.disable_warnings()

MODEL_NAME = "failure-classifier"
INFER_URL = "https://<model-inference-endpoint>"
TRUSTY_ROUTE = "https://<trustyai-service-route>"
TRUSTY_TOKEN = "<openshift-token>"

trusty_headers = {"Authorization": f"Bearer {TRUSTY_TOKEN}"}

## 2. Send Inference Requests

Send the test set to the deployed model. TrustyAI records each inference.

In [ ]:
X_test = pd.read_csv("test_data.csv")[data_utils.FEATURES].to_numpy(dtype="float64")

for start in range(0, len(X_test), 500):
    batch = X_test[start:start + 500]
    body = {
        "inputs": [
            {
                "name": "server_metrics",
                "shape": list(batch.shape),
                "datatype": "FP64",
                "data": batch.ravel().tolist(),
            }
        ]
    }
    response = requests.post(
        f"{INFER_URL}/v2/models/{MODEL_NAME}/infer", json=body, verify=False
    )
    response.raise_for_status()

print(f"Sent {len(X_test)} inference records")

# Give TrustyAI time to store the inference records
time.sleep(15)

## 3. Map the Field Names

Give the model fields readable names, so that the bias requests can use the column names.

In [ ]:
name_mapping = {
    "modelId": MODEL_NAME,
    "inputMapping": {
        f"server_metrics-{i}": column for i, column in enumerate(data_utils.FEATURES)
    },
    "outputMapping": {"output_0-0": "failure_pred"},
}

# Clear any previous mapping, so that this cell can run again
requests.delete(
    f"{TRUSTY_ROUTE}/info/names",
    data=MODEL_NAME,
    headers={**trusty_headers, "Content-Type": "application/json"},
    verify=False,
)

response = requests.post(
    f"{TRUSTY_ROUTE}/info/names", json=name_mapping, headers=trusty_headers, verify=False
)
print(response.status_code, response.text)

## 4. Compute the Bias Metrics

Compute SPD and DIR for each workload type, over the inference records of step 2.

In [ ]:
def bias_request(column):
    return {
        "modelId": MODEL_NAME,
        "protectedAttribute": column,
        "privilegedAttribute": 0.0,
        "unprivilegedAttribute": 1.0,
        "outcomeName": "failure_pred",
        "favorableOutcome": 0,
        "batchSize": len(X_test),
    }


for metric in ["spd", "dir"]:
    for column in data_utils.WORKLOAD_FEATURES:
        response = requests.post(
            f"{TRUSTY_ROUTE}/metrics/group/fairness/{metric}",
            json=bias_request(column),
            headers=trusty_headers,
            verify=False,
        )
        response.raise_for_status()
        result = response.json()
        print(
            f"{metric.upper():<4} {column:<24} {float(result['value']):>8.4f}"
            f"   outside bounds: {result['thresholds']['outsideBounds']}"
        )

## 5. Schedule the Bias Metrics

Schedule the metrics, so that TrustyAI computes them periodically and the OpenShift AI dashboard shows them in the **Model bias** tab of the model.

To list the scheduled metrics, use `GET /metrics/all/requests`.
To delete a scheduled metric, use `DELETE /metrics/group/fairness/<metric>/request` with the `{"requestId": "<id>"}` body.

In [ ]:
for metric in ["spd", "dir"]:
    for column in data_utils.WORKLOAD_FEATURES:
        response = requests.post(
            f"{TRUSTY_ROUTE}/metrics/group/fairness/{metric}/request",
            json=bias_request(column),
            headers=trusty_headers,
            verify=False,
        )
        response.raise_for_status()
        print(f"{metric.upper():<4} {column:<24} requestId: {response.json()['requestId']}")